# Ch.00 — Data Acquisition and Collection _(Exercise)_

> **The story:** Hadley Wickham formalised the principles of tidy data in 2014 — one variable per column, one observation per row, one value per cell. But tidy data assumes you already _have_ data. Before any of that, someone has to write the pipeline that gets raw records off a server, catches schema drift the moment it happens, and survives the API's rate limiter without corrupting anything. That someone is you, and this chapter is how you do it right.
>
> **Where you are:** This is Chapter 0 of the ML track — before any model exists. SmartVal AI is a future promise. Right now, the constraint is simpler: you cannot train a model on data you do not reliably have. This chapter builds the acquisition layer that makes every downstream chapter possible.
>
> **Notation:** `url` — API endpoint; `offset` — pagination cursor; `h` — content hash; `n` — retry attempt; `c` — base backoff (seconds); `ε` — jitter term.

---

## 0 · The Challenge

> **The mission**: SmartVal AI — build a reliable training corpus before any model can run

**What we know so far:**

- The SpaceX Falcon 9 launch history is the training vehicle for data acquisition skills
- **But we still can't reliably fetch the full corpus** — the API returns 10 records by default, not the 200+ needed

**What's blocking us:**
The v5 API's default pagination cap means a naive `requests.get(url).json()` silently returns a 10-record slice. A model trained on that slice trains on approximately 5% of available data — with no error, no warning, just a quietly degraded training set. On top of that, the SpaceX v3 → v5 migration renamed `launch_success` to `success`: any pipeline that does not validate schema at ingest reads `None` for every outcome label.

**What this chapter unlocks:**
A hash-gated, schema-validated, paginated pipeline that fetches all records idempotently and halts loudly on data drift — the foundation that makes every downstream training chapter reproducible.

Implement every function marked **TODO**. Each `raise NotImplementedError` is one stub.

| #   | Function                                        | Concept                         | Time   |
| --- | ----------------------------------------------- | ------------------------------- | ------ |
| 1   | `build_session()`                               | Retry + exponential backoff     | 10 min |
| 2   | `fetch_all_launches()`                          | Offset pagination               | 15 min |
| 3   | `LaunchRecord` + `fetch_validated_launches()`   | Pydantic schema validation      | 20 min |
| 4   | `fetch_with_change_detection()`                 | Content hashing                 | 10 min |
| 5   | `scrape_wikipedia_launches()`                   | BeautifulSoup + post-processing | 20 min |
| 6   | `handle_landing_outcome()`                      | MNAR treatment                  | 10 min |
| 7   | `coerce_boolean()` + `coerce_mass_kg()`         | Type normalisation              | 10 min |
| 8   | `merge_with_priority()` + `validate_launches()` | Source merge + contract         | 15 min |
| 9   | `run_pipeline()`                                | Idempotent assembly             | 15 min |


In [ ]:
# %pip install requests pydantic pandas beautifulsoup4 lxml --quiet

import hashlib, json, logging, re
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LAUNCHES_URL = "https://api.spacexdata.com/v5/launches"
RAW_JSONL = RAW_DIR / "launches.jsonl"
HASH_STORE = RAW_DIR / ".content_hashes.json"

print("Setup complete.", RAW_DIR, PROCESSED_DIR)

---

## § 1 — REST API Data Collection

You hit `GET /v5/launches`. You get a response. You assume you got all the launches. You did not.

The SpaceX v5 API defaults to **10** results per call and caps at **100** per page. Without explicit pagination, your "full dataset" is whatever launched most recently — not the 200+ record corpus you need to train on. No error. No warning. Just a subset that looks complete until you count it.

**Predict:** If you call `GET /v5/launches` with no parameters, how many records do you get back?

- (a) All launches — the API returns everything unless you limit it
- (b) 10 records — the API defaults to a small page
- (c) 0 records — you need to specify a date range

Run the preview cell below to find out.

### HTTP status codes

| Code    | Meaning      | Action                          |
| ------- | ------------ | ------------------------------- |
| 200     | OK           | Parse and persist               |
| 400     | Bad request  | Fix request; **do not retry**   |
| 429     | Rate limited | Back off; respect `Retry-After` |
| 500/503 | Server error | Retry with backoff              |

### Exponential backoff

When the server responds with 429 (rate limited), you need to wait before retrying. The right instinct is: wait longer after each failure, add some randomness so you don't sync up with other clients, and cap the wait so you don't stall forever.

That intuition — double the wait each attempt, add random jitter, cap at a max — is what `Retry(backoff_factor=1.0)` implements. Attempt 0: nearly instant. Attempt 1: ~2s. Attempt 2: ~4s. The jitter means no two clients retry at the exact same instant, so you don't replace one thundering herd with another.

> **Optional depth:** The formal expression: $t_{wait} = \min(c \cdot 2^n + \varepsilon,\ t_{max})$ — $n$ is the attempt number, $c=1\,\text{s}$ is the base, $\varepsilon \sim \text{Uniform}(0, c)$ is the jitter.

Reference: [data-acquisition.md](data-acquisition.md) § 1 — "Rate limiting and exponential backoff"


In [ ]:
def build_session(
    retries=5, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504)
):
    """
    TODO #1: Build a requests.Session with automatic retry and exponential backoff.

    Steps:
    1. Create requests.Session()
    2. Create Retry(total, backoff_factor, status_forcelist,
                    respect_retry_after_header=True)
    3. Create HTTPAdapter(max_retries=retry)
    4. session.mount("https://", adapter); session.mount("http://", adapter)
    5. Return session

    📖 data-acquisition.md § 1 — "Rate limiting and exponential backoff"
    """
    raise NotImplementedError("TODO #1: implement build_session()")


session = build_session()
print("Session:", session)

### Pagination — the silent truncation problem

The Predict cell above returned 10 records. There are 200+. The fix is an offset loop.

**Offset pattern:** `?offset=0&limit=100` → `?offset=100&limit=100` → ... → empty list = done.

Your task: implement `fetch_all_launches()` to keep fetching pages until the API returns an empty list, writing every record to JSONL as it goes.

**Warning:** `limit=100` is the API maximum. Requesting `limit=200` silently falls back to 100. After fetching, count the records in your JSONL — if the total equals a round multiple of 100, run one more page to confirm there's nothing left.

Reference: [data-acquisition.md](data-acquisition.md) § 1 — "Pagination: the silent truncation problem"


In [ ]:
def fetch_all_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """
    TODO #2: Full-refresh fetch to JSONL using offset pagination.

    Steps:
    1. sess = build_session()
    2. output.parent.mkdir(parents=True, exist_ok=True)
    3. Open output for writing ("w", encoding="utf-8")
    4. Loop:
       a. GET url with params={"offset": offset, "limit": limit}
       b. resp.raise_for_status()
       c. records = resp.json()
       d. if not records: break
       e. write each: f.write(json.dumps(record) + "\\n")
       f. written += len(records); offset += limit
    5. Return written

    📖 data-acquisition.md § 1 — "The complete fetch pattern"
    """
    raise NotImplementedError("TODO #2: implement fetch_all_launches()")


# Preview (does not require your implementation)
sample = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
print("Keys:", list(sample.keys()))

### Schema validation — catching drift at ingest time

Imagine you've been running a pipeline against SpaceX v3 for months. One day the team upgrades to v5 — and `launch_success` is now called `success`. Your pipeline still runs. No exception. Every record just quietly gets `None` for the target column instead of `True`/`False`. Your model trains on nulls and you don't find out until you check feature importances three days later.

The fix: validate every record against an expected schema _at ingest_, not downstream. Pydantic fires a `ValidationError` immediately on the first mismatched record, before anything gets written to disk.

Key patterns your implementation should use:

- `Optional[bool] = None` — null is legitimate for upcoming launches
- `@field_validator("date_utc", mode="before")` — transform raw string before validation
- `model_dump(mode="json")` — JSON-safe output for JSONL

Reference: [data-acquisition.md](data-acquisition.md) § 1 — "Schema validation: catching drift at ingest time"


In [ ]:
class LaunchRecord(BaseModel):
    """
    TODO #3a: Define Pydantic model for SpaceX v5 launch records.

    Fields: id:str  name:str  date_utc:datetime  success:Optional[bool]=None
            upcoming:bool  cores:list

    Add @field_validator("date_utc", mode="before"):
        if isinstance(v, str): return datetime.fromisoformat(v.rstrip("Z"))
        return v

    📖 data-acquisition.md § 1 — "Schema validation"
    """

    pass  # TODO: replace with fields + validator


def fetch_validated_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """
    TODO #3b: Like fetch_all_launches but validate each record with LaunchRecord.

    For each raw record:
      try:    launch = LaunchRecord(**raw); write launch.model_dump(mode="json")
      except ValidationError: log.warning(...); skipped += 1
    Return written count.

    📖 data-acquisition.md § 1 — "Schema validation"
    """
    # Hint:
    #   class LaunchRecord(BaseModel):
    #       """Pydantic model for SpaceX v5 launch records.
    #       ValidationError fires at ingest if a required field is absent — making
    #       schema drift a loud error rather than a silent null in the target column.
    raise NotImplementedError("TODO #3b: implement fetch_validated_launches()")


# Test against a live record
raw = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
try:
    v = LaunchRecord(**raw)
    print(f"id={v.id}  name={v.name}  success={v.success}  upcoming={v.upcoming}")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# Fetch all launches into a DataFrame for exploration
records, offset = [], 0
while True:
    batch = requests.get(
        LAUNCHES_URL, params={"offset": offset, "limit": 100}, timeout=15
    ).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_api = pd.DataFrame(
    [
        {
            "flight_number": r.get("flight_number"),
            "name": r.get("name"),
            "date_utc": r.get("date_utc"),
            "success": r.get("success"),
            "upcoming": r.get("upcoming"),
        }
        for r in records
    ]
)
df_api["date_utc"] = pd.to_datetime(df_api["date_utc"], utc=True)

print(f"Total launches : {len(df_api)}")
print(f"Success rate   : {df_api['success'].mean():.1%}  (excludes null)")
print(f"Null success   : {df_api['success'].isna().sum()}  (upcoming / unknown)")
df_api.head()

### What § 1 fixed — and what it still doesn't solve

Paginated, schema-validated fetching gives you a complete, consistently-shaped corpus for every launch from 2006 to today. The silent truncation is gone. Schema drift fires loudly at ingest.

**What's still missing:** the v5 API only covers flights from approximately 2013 onward. The early Falcon 9 test flights (2010–2012) predate the API. If your model needs to reason about early-program reliability — where failure rates were highest — those records simply aren't here.

The next section fixes this by scraping Wikipedia for pre-API launch history.


---

## § 2 — Web Scraping

The API gave you flights from 2013 onward. The early test flights — 2010, 2011, 2012 — are only on Wikipedia. You have no choice but to scrape.

The challenge with scraping is durability. A scraper that works today breaks the moment Wikipedia adds a column, splits a table, or changes a CSS class. Your job is to pick selectors that are stable by design rather than coincidence.

### Parsing hierarchy

```
URL → requests.get() → BeautifulSoup → .find("table", {"class":"wikitable"}) → pd.read_html()
```

`soup.find("table", {"class":"wikitable"})` is stable — `wikitable` is a MediaWiki convention that has existed for over a decade. Positional indexing (`soup.find_all("table")[3]`) is fragile — it breaks the moment any table is added before yours.

### Change detection

You check the hash of the raw HTML bytes on every run. On first run you store it. On every subsequent run, if the hash changed, you raise before parsing — a loud failure you can investigate, not a silent corruption that shows up weeks later in a model's predictions.

Reference: [data-acquisition.md](data-acquisition.md) § 2 — "Change detection: hash, don't assume"


In [ ]:
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"


def fetch_with_change_detection(url, name):
    """
    TODO #4: Fetch URL; raise ValueError if content hash changed since last run.

    Steps:
    1. requests.get(url, timeout=20, headers={"User-Agent":"research-bot/1.0"})
    2. resp.raise_for_status()
    3. current_hash = hashlib.sha256(resp.content).hexdigest()
    4. hashes = json.loads(HASH_STORE.read_text()) if HASH_STORE.exists() else {}
    5. last_hash = hashes.get(name)
    6. if last_hash and last_hash != current_hash: raise ValueError(...)
    7. hashes[name] = current_hash; HASH_STORE.write_text(json.dumps(hashes, indent=2))
    8. return resp.content

    📖 data-acquisition.md § 2 — "Change detection: hash, don't assume"
    """
    raise NotImplementedError("TODO #4: implement fetch_with_change_detection()")


print("fetch_with_change_detection() stub defined.")

In [ ]:
WIKI_RAW = RAW_DIR / "html" / "wiki_launches.html"
WIKI_RAW.parent.mkdir(parents=True, exist_ok=True)


def scrape_wikipedia_launches(url=WIKI_URL):
    """
     TODO #5: Fetch Wikipedia launch table; return cleaned DataFrame.

     Steps:
     1. html_bytes = fetch_with_change_detection(url, "wiki_falcon9_launches")
     2. WIKI_RAW.write_bytes(html_bytes)
     3. soup = BeautifulSoup(html_bytes, "html.parser")
     4. table = soup.find("table", {"class": "wikitable"})
        if table is None: raise RuntimeError(...)
     5. df = pd.read_html(str(table))[0]
     6. Flatten MultiIndex columns:
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [" ".join(filter(None, map(str, col))).strip() ...]
     7. df.replace(r"\\[.*?\\]", "", regex=True)   strip citations
     8. df.replace(r"^\\s*$", pd.NA, regex=True)     whitespace -> NA
     9. Keep rows where r.nunique(dropna=False) > 1     drop footnote rows
    10. df.dropna(how="all").reset_index(drop=True)
    11. Return df

     📖 data-acquisition.md § 2 — "Multi-header rows and the post-processing reality"
    """
    raise NotImplementedError("TODO #5: implement scrape_wikipedia_launches()")


# Uncomment after implementing TODO #4 and #5:
# df_wiki = scrape_wikipedia_launches()
# print(df_wiki.shape)
# df_wiki.head(3)
print("scrape_wikipedia_launches() stub defined.")

---

## § 3 — Data Wrangling

You have a raw API corpus and a Wikipedia table. They disagree on some values. Some fields are blank. Some blanks are fine; others are dangerous if you treat them the wrong way.

The dangerous case is `landing_outcome`. An absent value here does not mean the landing failed — it means no landing was attempted. If you impute `landing_success = False` for unattempted flights, you teach the model that those configurations failed. They simply weren't tried. That is a different thing, and conflating the two poisons the model's understanding of early-program choices.

### Missing value taxonomy: MCAR / MAR / MNAR

| Mechanism | Definition                                  | SpaceX example                                                             | Treatment                              |
| --------- | ------------------------------------------- | -------------------------------------------------------------------------- | -------------------------------------- |
| **MCAR**  | Missing independent of all variables        | 2% API dropout from timeouts                                               | Drop rows (< 5%) or impute with median |
| **MAR**   | Missing depends on _other observed_ columns | `payload_mass_kg` missing more for early launches (correlates with `year`) | Group imputation                       |
| **MNAR**  | Missing depends on _the value itself_       | `landing_outcome` absent when landing **not attempted**                    | Indicator columns — **do not impute**  |

**Warning:** MNAR is the category that kills model quality silently. Three correct states: attempted+succeeded, attempted+failed, not attempted. Your `handle_landing_outcome()` must produce all three — never collapse "not attempted" into "failed."

Reference: [data-acquisition.md](data-acquisition.md) § 3 — "Missing value taxonomy: MCAR, MAR, MNAR"


In [ ]:
def handle_landing_outcome(df):
    """
    TODO #6: MNAR treatment — split 'landing_outcome' into two indicator columns.

    Three states:
    - attempted+succeeded → landing_attempted=True,  landing_success=True
    - attempted+failed    → landing_attempted=True,  landing_success=False
    - not attempted       → landing_attempted=False, landing_success=NA

    Steps:
    1. df = df.copy()
    2. df["landing_attempted"] = df["landing_outcome"].notna()
    3. parse_outcome(v): pd.NA if isna(v); True if lower in success set;
       False if lower in failure set; else pd.NA   (not assumed failure)
    4. df["landing_success"] = df["landing_outcome"].map(parse_outcome).astype(pd.BooleanDtype())
    5. Return df

    📖 data-acquisition.md § 3 — MNAR section
    """
    raise NotImplementedError("TODO #6: implement handle_landing_outcome()")


demo = pd.DataFrame(
    {
        "flight": [1, 2, 3, 4, 5],
        "landing_outcome": ["Success", "Failure", None, "success", "Crash"],
    }
)
print(
    handle_landing_outcome(demo)[
        ["flight", "landing_outcome", "landing_attempted", "landing_success"]
    ]
)

In [ ]:
def coerce_boolean(series):
    """
    TODO #7a: Normalise any boolean-ish string to pd.BooleanDtype.

    True  ← 'true','yes','1','success','successful','landed'
    False ← 'false','no','0','failure','failed','crash','n/a','none','nan',''
    pd.NA ← anything else (unrecognised — not assumed)

    Pipeline: .astype(str).str.lower().str.strip().map(lambda...).astype(pd.BooleanDtype())

    📖 data-acquisition.md § 3 — "Type coercion: normalizing the chaos"
    """
    raise NotImplementedError("TODO #7a: implement coerce_boolean()")


def coerce_mass_kg(series):
    """
    TODO #7b: Parse '9525 kg', '9,525' to float.

    1. series.astype(str).str.replace(r'[^\\d.]', '', regex=True)
    2. pd.to_numeric(..., errors='coerce')   ← NaN for unparseable (visible failure)

    📖 data-acquisition.md § 3 — Type coercion table
    """
    raise NotImplementedError("TODO #7b: implement coerce_mass_kg()")


messy = pd.Series(["TRUE", "False", "1", "0", "Success", "N/A", None, "yes", "unknown"])
clean = coerce_boolean(messy)
print(pd.DataFrame({"raw": messy, "coerced": clean}).to_string(index=False))
print(f"\nNull (unrecognised): {clean.isna().sum()}")

### Deduplication and source-priority merging

The same launch can appear in the API and on Wikipedia with different values — one source was retroactively corrected after the other was scraped. You need to decide, per field, which source wins and document it.

`combine_first(other)` takes the left value when it's non-null, falls back to the right. API wins for outcome fields (more recent corrections); Wikipedia wins for descriptive text (older, more stable).

### Validation contract

Before writing anything to `data/processed/`, your pipeline must assert the invariants it promised. If any assertion fails, the pipeline halts with a clear error message — not a silent write of corrupt data downstream.

Reference: [data-acquisition.md](data-acquisition.md) § 3 — "Deduplication" + "Validation before downstream use"


In [ ]:
def merge_with_priority(api_df, csv_df, key="flight_number"):
    """
    TODO #8a: Outer join with explicit source priority per field.

    Steps:
    1. merged = api_df.merge(csv_df, on=key, suffixes=("_api","_csv"), how="outer")
    2. merged["success"] = merged["success_api"].combine_first(merged["success_csv"])
       drop "success_api", "success_csv"
    3. merged["launch_site"] = merged["launch_site_csv"].combine_first(merged["launch_site_api"])
       drop suffixed columns
    4. Return merged

    📖 data-acquisition.md § 3 — "Deduplication: exact vs near-duplicate"
    """
    raise NotImplementedError("TODO #8a: implement merge_with_priority()")


def validate_launches(df):
    """
    TODO #8b: Assert data quality invariants before writing to data/processed/.

    Assert:
    1. len(df) >= 50
    2. 'flight_number' and 'name' have zero nulls
    3. 'flight_number' has zero duplicates

    Each assert must carry a descriptive message with the actual value.

    📖 data-acquisition.md § 3 — "Validation before downstream use"
    """
    raise NotImplementedError("TODO #8b: implement validate_launches()")


try:
    validate_launches(df_api)
    print("Validation passed ✅")
except (AssertionError, NotImplementedError) as e:
    print(f"{type(e).__name__}: {e}")

---

## § 4 — Reproducible Pipeline

You now have all the pieces. The last question is whether running them in sequence produces the same result every time — even if you run it twice, three weeks apart, on a different machine.

Three things break reproducibility in data pipelines:

1. **Mutable sources** — the API returns different data on different days
2. **Local state** — files the pipeline reads that aren't in version control
3. **Implicit ordering** — step 3 silently reads stale intermediate data from a previous run

The fix is a disciplined separation:

```
data/
├── raw/        ← append-only; never overwritten
└── processed/  ← fully derivable from raw; safe to delete and regenerate
```

Hash-gated fetches (§ 2) make this idempotent: running twice fetches nothing new, writes nothing new, produces the same output.

Reference: [data-acquisition.md](data-acquisition.md) § 4 — "Idempotency" + "Raw vs processed"


In [ ]:
def run_pipeline():
    """
    TODO #9: Idempotent end-to-end pipeline.

    Stage 1  fetch:    fetched = fetch_validated_launches(output=RAW_JSONL)
    Stage 2  load:     df = pd.read_json(RAW_JSONL, lines=True)
                       log shape and null counts
    Stage 3  wrangle:  date_utc → datetime; success → coerce_boolean; dedup
    Stage 4  validate: validate_launches(df); raise on failure
    Stage 5  persist:  df.to_parquet(PROCESSED_DIR / "launches_clean.parquet")

    Log row counts at each stage. Return final DataFrame.

    📖 data-acquisition.md § 4 — "Logging and observability"
    """
    log.info("=== Pipeline start ===")
    raise NotImplementedError("TODO #9: implement run_pipeline()")


# Uncomment after implementing all TODOs:
# df_final = run_pipeline()
# df_final.head()
print("run_pipeline() stub defined.")

In [ ]:
# Fetch all and run wrangling inline (works even before TODOs are done)
records, offset = [], 0
while True:
    batch = requests.get(
        LAUNCHES_URL, params={"offset": offset, "limit": 100}, timeout=15
    ).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_check = pd.DataFrame(
    [
        {
            "flight_number": r.get("flight_number"),
            "name": r.get("name"),
            "date_utc": r.get("date_utc"),
            "success": r.get("success"),
        }
        for r in records
    ]
)
df_check["date_utc"] = pd.to_datetime(df_check["date_utc"], utc=True)

print(f"Records   : {len(df_check)}")
print(
    f"Date range: {df_check['date_utc'].min().date()} → {df_check['date_utc'].max().date()}"
)
print(f"Null success: {df_check['success'].isna().sum()}")
print()
# Uncomment after implementing coerce_boolean():
# df_check['success_bool'] = coerce_boolean(df_check['success'].astype(str))
# print('Success rate:', df_check['success_bool'].mean())
df_check.head()

---

## Summary

**Checkpoint:** When all TODOs pass, the § 0 challenge is resolved. You started with no training data and an API designed to hand you 10 records. You now have an idempotent pipeline that fetches the full 200+ record corpus, validates schema at ingest, detects source drift before it reaches your model, and handles MNAR fields correctly. `data/processed/launches_clean.parquet` is ready for Ch.01.

### Self-check

- [ ] `build_session()` — no `NotImplementedError`; session printed
- [ ] `fetch_all_launches()` — row count > 10 (more than page one)
- [ ] `LaunchRecord` — validates a live API record without error
- [ ] `fetch_with_change_detection()` — stores hash on first run; raises on content change
- [ ] `scrape_wikipedia_launches()` — DataFrame with flattened columns, no `[1]` markers
- [ ] `handle_landing_outcome()` — demo shows 3-column MNAR result correctly
- [ ] `coerce_boolean()` — `"unknown"` → `pd.NA`, not `True`/`False`
- [ ] `validate_launches()` — passes on `df_api` from the summary cell
- [ ] `run_pipeline()` — writes `data/processed/launches_clean.parquet`

### What these skills unlock downstream

| What you built                          | Used in                                                                |
| --------------------------------------- | ---------------------------------------------------------------------- |
| `data/processed/launches_clean.parquet` | Input to `exercises/01-ml/01-regression/src/data-prep.py`              |
| `validate_launches()` pattern           | Template for `data-prep.py` exercises #11–12 (PSI / KS drift tests)    |
| `coerce_boolean()`                      | Extended in `data-prep.py` for SmartVal AI California Housing dataset  |
| MNAR indicator pattern                  | Applied to every target-adjacent field in supervised learning chapters |

**Forward:** The validation contract you wrote here is the same pattern Ch.01 applies to the California Housing dataset before training. Every time `loss.backward()` eventually runs on clean data, it's because a pipeline upstream was paranoid about what it was handing downstream.

**Solution:** [notebook-solution.ipynb](notebook-solution.ipynb)
